# Testing Queries

Note: For now, we are lucky that the imports work, because we are using relative imports, and at one point, it will cause errors.

## IMPORTS

In [1]:
import os
import sys
import random
import requests
import argparse
from typing import List, Set

import numpy as np
import pandas as pd

In [2]:
from db.yagodb import YagoDB
from db.constants.main import YAGO_ALL_ENTITY_COUNT, YAGO_FACTS_ENTITY_COUNT
from db.functions.entity import get_random_entities_query

In [3]:
from kg.query import get_triples_multiple_subjects_query, get_description_multiple_entities_query, query_kg, get_triples_from_response

In [4]:
from utils.constants import YAGO_ENTITY_STORE_DB_PATH, YAGO_PREFIXES_PATH, YAGO_ENDPOINT_URL
from utils.prefix import get_prefixes, get_url_from_prefix_and_id
from utils.random_walk2 import SPARQL_COLUMNS_DICT, RandomWalk2

### CONSTANTS

### FUNCTIONS

### Experiment Single Walks

In [5]:
YAGO_ENTITY_STORE_DB_PATH

'/home/ubuntu/ClaimBenchKG/yago/yago_all.db'

In [6]:
yago_db = YagoDB(YAGO_ENTITY_STORE_DB_PATH)

In [7]:
result = yago_db.query("""SELECT item_id, item_label, count FROM items 
                       WHERE item_label in ('http://yago-knowledge.org/resource/Alpaca_Q58244340')
                       ORDER BY count""")
print(result)

[('yago:Alpaca_Q58244340', 'http://yago-knowledge.org/resource/Alpaca_Q58244340', 10)]


In [8]:
print("Germany\\'s national railway company")

Germany\'s national railway company


In [9]:
# Get entities with the specific labels and use the entity with the highest degree
data = query_kg(YAGO_ENDPOINT_URL, """
        PREFIX yago: <http://yago-knowledge.org/resource/>\n
        PREFIX schema: <http://schema.org/>\n
        PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>\n
        PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>\n
        PREFIX wd: <http://www.wikidata.org/entity/>\n
        SELECT DISTINCT ?entity ?label WHERE {
            VALUES ?label { '''Alpaca'''@en 'Elvis Presley'@en '''Germany's national railway company'''@en }
                {
                    ?entity rdfs:label ?label
                } UNION {
                    ?entity schema:alternateName ?label
                }
        }
""")

results = data["results"]["bindings"]

entity_set = {results["entity"]["value"] for results in results}

entity_counts = yago_db.query(f"""SELECT item_label, count FROM items 
                       WHERE item_label in ({', '.join([f"'{entity}'" for entity in entity_set])})
                       ORDER BY count DESC""")
entity_counts_dict = {result[0]: result[1] for result in entity_counts}
print(entity_counts_dict)

entities = {results["label"]["value"]: None for results in results}
for result in results:
    entity = result["entity"]["value"]
    label = result["label"]["value"]
    if entities[label] is None:
        entities[label] = entity
    else:
        if (entity in entity_counts_dict) and (entity_counts_dict[entity] > entity_counts_dict[entities[label]]):
            entities[label] = entity

# print(results)
# entities = {results["label"]["value"]: results["entity"]["value"] for results in results}
print(entities)

{'http://yago-knowledge.org/resource/Elvis_Presley': 277, 'http://yago-knowledge.org/resource/Elvis_Presley__u0028_album_u0029_': 35, 'http://yago-knowledge.org/resource/Alpaca_Q58244340': 10, 'http://yago-knowledge.org/resource/Alpaca_Q66124520': 7, 'http://yago-knowledge.org/resource/Elvis_Presley_Q47509719': 7, 'http://yago-knowledge.org/resource/Alpaca_Q98505015': 6}
{'Alpaca': 'http://yago-knowledge.org/resource/Alpaca_Q58244340', 'Elvis Presley': 'http://yago-knowledge.org/resource/Elvis_Presley'}


In [10]:
# Get entities with the specific labels and use the entity with the highest degree
data = query_kg(YAGO_ENDPOINT_URL, """
        PREFIX yago: <http://yago-knowledge.org/resource/>\n
        PREFIX schema: <http://schema.org/>\n
        PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>\n
        PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>\n
        PREFIX wd: <http://www.wikidata.org/entity/>\n
        SELECT DISTINCT ?entity ?qid WHERE {
            VALUES ?qid { wd:Q752297 wd:Q2578249 }
                {
                    ?entity owl:sameAs ?qid
                }
        }
""")

results = data["results"]["bindings"]

print(results)

[{'entity': {'type': 'uri', 'value': 'http://yago-knowledge.org/resource/Doctor_of_Philosophy'}, 'qid': {'type': 'uri', 'value': 'http://www.wikidata.org/entity/Q752297'}}, {'entity': {'type': 'uri', 'value': 'http://yago-knowledge.org/resource/Cabinet_of_Israel'}, 'qid': {'type': 'uri', 'value': 'http://www.wikidata.org/entity/Q2578249'}}]


In [17]:
# Get entities with the specific labels and use the entity with the highest degree
data = query_kg(YAGO_ENDPOINT_URL, """
        PREFIX yago: <http://yago-knowledge.org/resource/>\n
        PREFIX schema: <http://schema.org/>\n
        PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>\n
        PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>\n
        PREFIX wd: <http://www.wikidata.org/entity/>\n
        SELECT DISTINCT ?r WHERE {
                {
                    yago:Caribbean_Sea ?r ?o
                }
        }
""")

results = data["results"]["bindings"]

import json
print(json.dumps(results))

[{"r": {"type": "uri", "value": "http://schema.org/alternateName"}}, {"r": {"type": "uri", "value": "http://schema.org/geo"}}, {"r": {"type": "uri", "value": "http://schema.org/image"}}, {"r": {"type": "uri", "value": "http://schema.org/location"}}, {"r": {"type": "uri", "value": "http://schema.org/mainEntityOfPage"}}, {"r": {"type": "uri", "value": "http://schema.org/sameAs"}}, {"r": {"type": "uri", "value": "http://yago-knowledge.org/resource/area"}}, {"r": {"type": "uri", "value": "http://www.w3.org/1999/02/22-rdf-syntax-ns#type"}}, {"r": {"type": "uri", "value": "http://www.w3.org/2000/01/rdf-schema#comment"}}, {"r": {"type": "uri", "value": "http://www.w3.org/2000/01/rdf-schema#label"}}, {"r": {"type": "uri", "value": "http://www.w3.org/2002/07/owl#sameAs"}}]


In [30]:
query_kg(YAGO_ENDPOINT_URL, """
        PREFIX yago: <http://yago-knowledge.org/resource/>\n
        PREFIX schema: <http://schema.org/>\n
        PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>\n
        PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>\n
        PREFIX wd: <http://www.wikidata.org/entity/>\n
        SELECT ?o WHERE 
        {
            {
                yago:Belgium rdfs:label ?o
                FILTER (lang(?o) = 'en')
            } UNION {
                yago:Belgium owl:sameAs ?o
            }
            BIND(IF(ISIRI(?o), 2, 1) AS ?order)
        } 
        ORDER BY ?order
        LIMIT 5""")

{'head': {'vars': ['o']},
 'results': {'bindings': [{'o': {'xml:lang': 'en',
     'type': 'literal',
     'value': 'Belgium'}},
   {'o': {'type': 'uri', 'value': 'http://www.wikidata.org/entity/Q31'}}]}}

In [14]:
query_kg(YAGO_ENDPOINT_URL, """
        PREFIX yago: <http://yago-knowledge.org/resource/>\n
        PREFIX schema: <http://schema.org/>\n
        SELECT ?tailEntity WHERE {yago:Belgium yago:leader ?tailEntity}
""")

{'head': {'vars': ['tailEntity']},
 'results': {'bindings': [{'tailEntity': {'type': 'uri',
     'value': 'http://yago-knowledge.org/resource/Philippe_of_Belgium'}},
   {'tailEntity': {'type': 'uri',
     'value': 'http://yago-knowledge.org/resource/Alexander_De_Croo'}}]}}

In [50]:
PREFIX_STRING = """
PREFIX yago: <http://yago-knowledge.org/resource/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
PREFIX ontolex: <http://www.w3.org/ns/lemon/ontolex#>
PREFIX dct: <http://purl.org/dc/terms/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX owl: <http://www.w3.org/2002/07/owl#>
PREFIX wikibase: <http://wikiba.se/ontology#>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
PREFIX schema: <http://schema.org/>
PREFIX cc: <http://creativecommons.org/ns#>
PREFIX geo: <http://www.opengis.net/ont/geosparql#>
PREFIX prov: <http://www.w3.org/ns/prov#>
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX data: <https://www.wikidata.org/wiki/Special:EntityData/>
PREFIX sh: <http://www.w3.org/ns/shacl#>
PREFIX s: <http://www.wikidata.org/entity/statement/>
PREFIX ref: <http://www.wikidata.org/reference/>
PREFIX v: <http://www.wikidata.org/value/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX wpq: <http://www.wikidata.org/prop/quant/>
PREFIX wdtn: <http://www.wikidata.org/prop/direct-normalized/>
PREFIX p: <http://www.wikidata.org/prop/>
PREFIX ps: <http://www.wikidata.org/prop/statement/>
PREFIX psv: <http://www.wikidata.org/prop/statement/value/>
PREFIX psn: <http://www.wikidata.org/prop/statement/value-normalized/>
PREFIX pq: <http://www.wikidata.org/prop/qualifier/>
PREFIX pqv: <http://www.wikidata.org/prop/qualifier/value/>
PREFIX pqn: <http://www.wikidata.org/prop/qualifier/value-normalized/>
PREFIX pr: <http://www.wikidata.org/prop/reference/>
PREFIX prv: <http://www.wikidata.org/prop/reference/value/>
PREFIX prn: <http://www.wikidata.org/prop/reference/value-normalized/>
PREFIX wdno: <http://www.wikidata.org/prop/novalue/>
PREFIX ys: <http://yago-knowledge.org/schema#>
"""

In [52]:
sparql_head_relations = """\n%s\nSELECT ?relation\nWHERE {\n %s ?relation ?x .\n}"""
entity_id = " m.03_dwn"
sparql_relations_extract_head = sparql_head_relations % (PREFIX_STRING, entity_id)
sparql_relations_extract_head

'\n\nPREFIX yago: <http://yago-knowledge.org/resource/>\nPREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>\nPREFIX xsd: <http://www.w3.org/2001/XMLSchema#>\nPREFIX ontolex: <http://www.w3.org/ns/lemon/ontolex#>\nPREFIX dct: <http://purl.org/dc/terms/>\nPREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>\nPREFIX owl: <http://www.w3.org/2002/07/owl#>\nPREFIX wikibase: <http://wikiba.se/ontology#>\nPREFIX skos: <http://www.w3.org/2004/02/skos/core#>\nPREFIX schema: <http://schema.org/>\nPREFIX cc: <http://creativecommons.org/ns#>\nPREFIX geo: <http://www.opengis.net/ont/geosparql#>\nPREFIX prov: <http://www.w3.org/ns/prov#>\nPREFIX wd: <http://www.wikidata.org/entity/>\nPREFIX data: <https://www.wikidata.org/wiki/Special:EntityData/>\nPREFIX sh: <http://www.w3.org/ns/shacl#>\nPREFIX s: <http://www.wikidata.org/entity/statement/>\nPREFIX ref: <http://www.wikidata.org/reference/>\nPREFIX v: <http://www.wikidata.org/value/>\nPREFIX wdt: <http://www.wikidata.org/prop/direct/>\nPREFIX